# Lab 05 — Estabilidade em malha fechada: Routh, lugar das raízes e margens

**Unidade II — Elementos essenciais em um sistema de controle** · conteúdo 2.1 do PPC

**Objetivos:**
1. Determinar a faixa de ganho estabilizante por Routh–Hurwitz e confirmar numericamente;
2. Ler o lugar das raízes: ganho crítico $K_u$ e frequência crítica $\omega_u$;
3. Medir margens de ganho e de fase e relacioná-las com o comportamento temporal;
4. Preparar os dados ($K_u$, $T_u$) que a sintonia de Ziegler–Nichols usará na Unidade IV.

**Referências:** Åström & Murray (FBS), caps. 10 e 12 · Ogata, caps. 5 e 8 · Dorf & Bishop, caps. 6–9.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import control as ct
    print("python-control", ct.__version__)
except ImportError:
    %pip install control
    import control as ct

## 1. Planta do laboratório

Usaremos uma planta de 3ª ordem — a menor ordem em que o ganho proporcional pode instabilizar:
$$G(s) = \frac{1}{(s+1)(s+2)(s+4)} \qquad L(s) = K\,G(s)$$

In [ ]:
G = ct.tf([1], np.polymul(np.polymul([1, 1], [1, 2]), [1, 4]))
print("G(s) =", G)
print("Polos de malha aberta:", ct.poles(G))

## 2. Routh–Hurwitz analítico

Polinômio característico de malha fechada: $1 + KG(s) = 0$
$$s^3 + 7s^2 + 14s + (8 + K) = 0$$
Para 3ª ordem $s^3 + a_2 s^2 + a_1 s + a_0$: estável ⟺ todos os coeficientes > 0 **e** $a_2 a_1 > a_0$:
$$7 \cdot 14 > 8 + K \iff K < 90 \qquad\text{e}\qquad 8 + K > 0 \iff K > -8$$
**Faixa estabilizante: $-8 < K < 90$.** No limiar $K_u = 90$, a linha de $s^1$ zera e o sistema
oscila com $\omega_u = \sqrt{a_1} = \sqrt{14} \approx 3{,}74$ rad/s.

In [ ]:
# confirmação numérica: varredura de K e verificação dos polos de malha fechada
Ks = np.linspace(1, 120, 400)
estavel = []
for Kv in Ks:
    T_cl = ct.feedback(Kv * G, 1)
    estavel.append(np.all(ct.poles(T_cl).real < 0))
estavel = np.array(estavel)

K_u_num = Ks[np.argmax(~estavel)]  # primeiro K instável
print(f"Ganho crítico numérico: K_u ≈ {K_u_num:.1f} (analítico: 90)")

plt.figure(figsize=(9, 2.2))
plt.fill_between(Ks, 0, 1, where=estavel, color='C2', alpha=0.5, label='estável')
plt.fill_between(Ks, 0, 1, where=~estavel, color='C3', alpha=0.5, label='instável')
plt.yticks([])
plt.xlabel('Ganho K')
plt.title('Varredura de estabilidade (confirma Routh–Hurwitz)')
plt.legend(loc='center left')
plt.show()

## 3. Lugar das raízes

O lugar das raízes mostra a trajetória dos polos de malha fechada quando $K: 0 \to \infty$.
O cruzamento do eixo imaginário deve ocorrer exatamente em $K_u = 90$, $\omega_u = 3{,}74$ rad/s.

In [ ]:
plt.figure(figsize=(8, 6))
ct.root_locus_plot(G)
plt.title('Lugar das raízes de L(s) = K·G(s)')
plt.show()

**Leitura do gráfico:** dois ramos partem dos polos em −1 e −2, encontram-se e cruzam para o
semiplano direito; o terceiro ramo vai de −4 para −∞. Clique/inspeção no cruzamento do eixo
imaginário confirma $\omega \approx 3{,}74$ e $K \approx 90$.

## 4. Resposta temporal nas três regiões

In [ ]:
t = np.linspace(0, 15, 1500)
plt.figure(figsize=(9, 5))
for Kv, nome in [(20, 'K = 20 (estável)'),
                 (90, 'K = 90 = Ku (marginal)'),
                 (120, 'K = 120 (instável)')]:
    T_cl = ct.feedback(Kv * G, 1)
    resp = ct.step_response(T_cl, t)
    plt.plot(resp.time, resp.outputs, lw=1.8, label=nome)
plt.axhline(1, color='gray', ls='--')
plt.ylim(-1, 3)
plt.xlabel('Tempo [s]'); plt.ylabel('y(t)')
plt.title('As três regiões de estabilidade')
plt.legend(); plt.grid(True)
plt.show()

# período da oscilação sustentada em K = Ku
T_u = 2 * np.pi / np.sqrt(14)
print(f"Período crítico: T_u = 2*pi/omega_u = {T_u:.2f} s")
print(">>> Anotar (K_u, T_u): serão a entrada da sintonia de Ziegler–Nichols no Lab 09! <<<")

## 5. Margens de ganho e de fase

As margens medem a **distância da instabilidade** no domínio da frequência — mais informativas
que o simples "estável/instável".

In [ ]:
K_projeto = 20
L = K_projeto * G
gm, pm, wcg, wcp = ct.margin(L)
print(f"Para K = {K_projeto}:")
print(f"  Margem de ganho GM = {gm:.2f} ({20*np.log10(gm):.1f} dB) na frequência {wcg:.2f} rad/s")
print(f"  Margem de fase  PM = {pm:.1f} graus na frequência {wcp:.2f} rad/s")
print(f"  Verificação: K * GM = {K_projeto * gm:.1f} (deve ser o ganho crítico 90)")

plt.figure(figsize=(9, 6))
ct.bode_plot(L, dB=True, display_margins=True)
plt.show()

## 6. Margem de fase × sobressinal

Regra prática: $\zeta \approx \mathrm{PM}/100$ (PM em graus, válida para PM ≲ 60°).
Vamos verificar variando K:

In [ ]:
print(f"{'K':>5s} | {'PM [graus]':>10s} | {'Mp [%]':>7s} | {'zeta~PM/100':>11s}")
print('-' * 45)
for Kv in [5, 10, 20, 40, 60]:
    L = Kv * G
    _, pm, _, _ = ct.margin(L)
    info = ct.step_info(ct.feedback(L, 1))
    print(f"{Kv:5d} | {pm:10.1f} | {info['Overshoot']:7.1f} | {pm/100:11.2f}")

Quanto menor a margem de fase, maior o sobressinal — mesmo permanecendo estável.
**Critério de projeto do curso: PM ≥ 45° e GM ≥ 6 dB.**

## 7. Diagrama de Nyquist (visão complementar)

In [ ]:
plt.figure(figsize=(7, 6))
ct.nyquist_plot(20 * G)
plt.title('Nyquist de L(s) = 20·G(s): distância ao ponto crítico −1')
plt.show()

A distância mínima da curva ao ponto $-1$ resume a robustez (é o inverso do pico da função
sensibilidade). Se a curva envolver $-1$, a malha fechada é instável (critério de Nyquist).

### 7.1 O critério de Nyquist completo: $Z = N + P$

Enunciado formal (Åström & Murray, cap. 10 — apresentado no CDS 110/Caltech como a ferramenta
central de análise de malha):

> Seja $L(s) = P(s)C(s)$ a FT de malha. Definindo
> $P$ = nº de polos de $L$ no semiplano direito (SPD),
> $N$ = nº de envolvimentos **horários** do ponto $-1$ pela curva de Nyquist,
> $Z$ = nº de zeros de $1 + L$ no SPD (= polos instáveis de malha fechada),
> vale $\boxed{Z = N + P}$. Malha fechada estável $\iff Z = 0$.

Com `ct.nyquist_response` obtemos $N$ automaticamente e auditamos o critério:

In [ ]:
L20 = 20 * G
nyq = ct.nyquist_response(L20)
N = nyq.count
P_rhp = np.sum(np.real(L20.poles()) > 0)
Z = np.sum(np.real((1 + L20).zeros()) > 0)
print(f"N (envolvimentos de -1) = {N}")
print(f"P (polos SPD de L)      = {P_rhp}")
print(f"Z = N + P               = {N + P_rhp}  (verificado: {Z})")
print("Malha fechada estável!" if N + P_rhp == 0 else "Malha fechada INSTÁVEL!")

### 7.2 Planta instável em malha aberta: quando o envolvimento é OBRIGATÓRIO

Ideia contraintuitiva com consequência direta no projeto final: o controle de POSIÇÃO do
kit tem planta com polo na origem, o caso-limite em que o contorno de Nyquist precisa ser
deformado para contorná-lo — e o raciocínio Z = N + P é o único confiável. Se a planta tem
$P > 0$ polos instáveis, a curva de Nyquist **precisa** envolver $-1$ exatamente $N = -P$
vezes (anti-horário) para que $Z = 0$. Exemplo do CDS 110: pêndulo invertido normalizado
$$P_{pend}(s) = \frac{1}{s^2 + 0{,}1s - 1}$$
(um polo instável) com controlador PD $C(s) = k_d s + k_p$:

In [ ]:
P_pend = ct.tf([1], [1, 0.1, -1])
print("Polos da planta:", P_pend.poles())   # um deles no SPD

C_pd = ct.tf([2, 10], [1])                  # kd = 2, kp = 10
L_pend = P_pend * C_pd

plt.figure(figsize=(7, 6))
ct.nyquist_plot(L_pend)
plt.title('Pêndulo invertido + PD: envolvimento anti-horário estabiliza')
plt.show()

nyq2 = ct.nyquist_response(L_pend)
P2 = np.sum(np.real(L_pend.poles()) > 0)
print(f"N = {nyq2.count}, P = {P2}  =>  Z = {nyq2.count + P2}")
print("Estável em malha fechada!" if nyq2.count + P2 == 0 else "Instável!")

# confirmação no domínio do tempo
T_pend = ct.feedback(L_pend, 1)
print("Polos de malha fechada:", T_pend.poles())

**Moral:** para plantas instáveis, "afastar-se de $-1$" não é a meta — envolver $-1$ do jeito
certo é. Bode e margens clássicas podem enganar nesses casos; Nyquist não.

### 7.3 A terceira margem: margem de estabilidade $s_m$

GM e PM medem perturbações *puras* de ganho ou fase. A **margem de estabilidade**
$s_m = \min_\omega |1 + L(j\omega)|$ (a menor distância ao ponto $-1$) captura perturbações
combinadas e vale $s_m = 1/M_s$, ligando Nyquist ao pico de sensibilidade do Lab 08:

In [ ]:
resp_L = ct.frequency_response(L20, np.logspace(-2, 2, 2000))
dist = np.abs(1 + resp_L.complex.flatten())
sm = dist.min()
w_sm = resp_L.omega[np.argmin(dist)]
print(f"s_m = {sm:.3f} em w = {w_sm:.2f} rad/s  =>  Ms = 1/s_m = {1/sm:.2f}")
print("Criterio do curso: Ms <= 2  <=>  s_m >= 0.5")

---
> **🖼️ Figuras de apoio nos livros:**
> - Nise, **Tabela 6.3** — tabela de Routh completa do Exemplo 6.1 (modelo de preenchimento passo a passo). Cap. 6, p. 456 do arquivo PDF (a cópia digital não exibe o nº impresso).
> - Ogata, **Figura 6.3** — sistema do Exemplo 6.1 e a construção completa do seu lugar das raízes na sequência. Cap. 6 (Exemplo 6.1), **p. 249** (p. 260 do PDF).
> - Ogata, **Figura 7.67** — definição gráfica das margens de ganho e de fase (sistemas estáveis × instáveis). Cap. 7, **p. 426** (p. 437 do PDF).
> - Ogata, **Figura 7.47** — contorno de Nyquist no plano $s$ (eixo $j\omega$ + semicírculo de raio infinito). Cap. 7, **p. 411** (p. 422 do PDF).
> - Transparências CDS 110 **L7-2**, **slide 5** — enunciado do critério de Nyquist ($Z = N + P$) com o contorno D.
> - Transparências CDS 110 **L7-2**, **slide 10** — GM e PM no Nyquist e no Bode, lado a lado.

## Exercícios (relatório do Lab 05)

**E1.** Para $L(s) = \dfrac{K}{s(s+1)(s+5)}$, determine por Routh a faixa estabilizante de $K$,
o ganho crítico e $\omega_u$. Confirme com lugar das raízes e varredura numérica.

**E2.** No sistema do E1, encontre (por busca) o valor de $K$ que dá PM = 45°. Meça $M_p$ e
$t_s$ da malha fechada resultante.

**E3.** Adicione tempo morto de 0,3 s à planta desta aula (`ct.pade(0.3, 3)`) e meça as novas
margens para K = 20. Quanto o tempo morto "custou" de margem de fase? Estime também pelo valor
teórico $\Delta\phi = \omega_{cp}\,\theta$ (em rad).

**E4.** Trace o lugar das raízes de $L(s) = K\dfrac{s+3}{s(s+1)(s+5)}$ (zero adicionado) e
compare com o E1. O que o zero fez com os ramos? Relacione com a ação derivativa do PID.

**E5.** Para o pêndulo invertido da seção 7.2, tente estabilizar com controlador **apenas
proporcional** $C = k_p$. Use Nyquist ($Z = N + P$) para mostrar que nenhum $k_p$ estabiliza,
e explique o porquê (dica: fase de $L$ nunca cruza $-180°$ com o envolvimento correto sem a
fase adiantada do termo derivativo).

In [ ]:
# E1 — sua solução aqui

In [ ]:
# E2 — sua solução aqui

In [ ]:
# E3 — sua solução aqui

In [ ]:
# E4 — sua solução aqui

In [ ]:
# E5 — sua solução aqui